# Supervised visual fine tuning in a native GPU workspace

This experiment updates the final one/two official CLIP vision blocks and a
small point-guided MIL head. Fixed fits train on34 recordings;12 validation
recordings select the model and threshold. The previous12-recording mixed test
has already been examined and is explicitly development data here.

Use a private Kaggle input containing the separate source and framepack ZIPs, or
upload the private framepack in Colab. Enable a GPU. All1879 sampled times and
3758 RGB224 images stay in native Python; no descriptions, LLM APIs or audio are
used. The opposite custom camera family remains excluded everywhere. These
within-filming-batch results cannot establish unseen-session80%/87% accuracy.


In [ ]:
# Restore exact source only; no private frame decoding or AI API calls.
import hashlib, io, json, os, shutil, subprocess, sys, zipfile
from pathlib import Path
BASE = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path('/content') if Path('/content').is_dir() else Path.cwd()
INPUT = Path('/kaggle/input') if Path('/kaggle/input').is_dir() else BASE
WORKSPACE = BASE/'datathon_visual_workspace'
WORKSPACE.mkdir(parents=True, exist_ok=True)
SOURCE_MEMBERS = {'datathon_ml/__init__.py': '52a650dee180328e591734380a92d44cbf9c5b676eb6b0e2ca58eab971e1a1a8', 'datathon_ml/boosted.py': '2ef7aa4fc14ec65afd6578965abe06f8e325de28a4e29722b0cede50e5e598d8', 'datathon_ml/clip_features.py': '2c46c9de2c92c90129949fb9e51fe7a4e98a58a23aa6e783b644a8b3004f05c2', 'datathon_ml/custom_annotation.py': 'd6759f70263d0c45b2f7d8fc42bf75fc1db053247a212d7ae35b47f00418f7f7', 'datathon_ml/custom_training.py': '55dd80c3e086b9a028e89e1e44bc46262f3c44d70488d01133af67ef1f9749e2', 'datathon_ml/data.py': '773bbe9c0404ff7703439e0924cd628f8bd60e35e6f382d85cab2122dc6fd0b3', 'datathon_ml/features.py': '5cb16dc5566874fb874832546d7b68529215b1c341cfdcd6e68aeb43e9cb350c', 'datathon_ml/finetune_visual.py': 'a1bff5b688873ce11b8ee668fb9648d92f5276b082d29a5361f1d181777dd7ce', 'datathon_ml/local_vlm.py': '1e767b8b79a78a69d42e2d77895ff7c88fa3c32255989f82c6cedc8269952ff2', 'datathon_ml/mil.py': '9331b04606c9add5595c0fc5ce24d22b637f47dbc35271f5950599b55e438110', 'datathon_ml/model.py': '9eaa61e54faaf405a4b0ea61a5c7a7de3d3245ddafae7c0de930ce05638bdb71', 'datathon_ml/motion.py': 'da2e739f3d7b651afd7d2afadc9e1bde2b163e1cb749b04e2183ce2a411432d5', 'datathon_ml/privileged_notes.py': 'd6c4e84c25f466db263941c7f391d7db829750b0e5831d2c0755f46ae1cd9631', 'datathon_ml/reviewed_models.py': '03fab2030e97f3a197dbd0a16db314ca6ec96b2228407d4ad056d986ec898a12', 'datathon_ml/scene_temporal.py': '18916f76767cbaa803b5e5b45564ce1f56456373f6ae70e2f1942e3adf2fe688', 'datathon_ml/semantic_clip.py': '304ec825c10b1ae5afc01b8cff6ffac9c32f05170733ad6747863c53c7c3c834', 'datathon_ml/semantic_heads.py': '8127b8eb47bb898290b54a5c143ede40d73154d1ea778e5b5f1c9a3ca9150de1', 'datathon_ml/semantic_vlm.py': '05290f452edc482ae8f047ffc78c9c3c10159fa80a37f0564398dadb018bbb2c', 'datathon_ml/temporal.py': '8fc78c56f43eeaf21bd951a35fd5674703a0f6028dc6d5fa86e0a9d4d4bee74e', 'datathon_ml/unlabeled_adaptation.py': 'd25bedc270fb586c40f4495b77a0117e2a17aa9dca2c4d68a609b9a681cc07a3', 'finetune_experiment.py': '4f144a65781e41c1f55e0541c0be40bab3a5f0dae0963657e0510d216c9116e4', 'reference/clip_baseline_cv.json': '8e716080485cfc78a593bca7cb87454e764d7b843c7a7d45dccbf0533794d085', 'requirements-experiments.txt': '4849c266174ecb7ef92ea7b3151d44f52b37f2ca1983248e037c6b5e877f1408', 'requirements-ml.txt': '55fb43ca08f740ef1d51e663349f2c20a240830c766d99bea5ceafd4510df015', 'reviewed_experiment.py': '5ee4accea1f6eae50d959871d68f8f5cfe2c8d6134099ccf379b69920afcff6f', 'tests/test_boosted.py': '25db0ec078d6bb2d742eb0d61737916eefc44826d0efcc4ccfba185d723ed108', 'tests/test_camera_pairs.py': '20ef27e609b258aab54aa17a846ee230739a4dfab1999956839b8aea64f6663e', 'tests/test_clip_features.py': '2bd7a988188c4bc1e2134ae755b5da6ee8338deebc7f26a874def8d826527ced', 'tests/test_custom_annotation.py': '6d6b6e7dd752edfc5756e33d05ec05da4fafa229eb1eae5b2bac1aba8852679d', 'tests/test_custom_annotation_detail.py': '7056265aa219183fe6c771eaae4abe269b5a9f31860180b0025b752a2e7e2a54', 'tests/test_custom_manifest.py': 'd8dae9d5b5b02ae0b2ec523c072b73a1f4b64ec2863e5726d5ff5da6ce691026', 'tests/test_custom_training.py': '2a6792c9cbc09515869ad26f6cf8a2cdb52c35f4c8ad4b7cabb24be8ef5b2e21', 'tests/test_data.py': 'b720f4a82ae538345c94cd32f8315ea49c2587a48ba6cbf0a167e79e02945ef3', 'tests/test_experiment.py': '2baac161cf78f683c92855f8b4b726b4071e2720bcd972a3ec61c3a9e6d637f4', 'tests/test_features.py': '4807486e9aa1ed86a3eca54aefeed28148188935d6c84154865c0790a339b29f', 'tests/test_finetune_visual.py': 'fb7c69015ea896af93e5cc6d9a33fe2c05347a8b5c25a310974cbf304cf1e202', 'tests/test_ingest_reviewed_workbook.py': '2adc9052d518aef42fe70ae8ab8cd424005af839a4b1848bb33c1f964ed68e7f', 'tests/test_local_vlm.py': 'b11e32adc2a0ca4d586680ecf097677dfb22e755cfa3d765252e37758854f230', 'tests/test_mil.py': 'a7091a45d7f00f8b1e6796cd0dd01dcf18fea326d7582d2686555d3ccd229887', 'tests/test_model.py': '415f7c6ef968f689bae098186b6894b0a50f8790f52151eef003cca7c0b6c6e3', 'tests/test_motion.py': '983ced3dfc23d1734be4eb621571ba1d1a3973a4e6f07ec76bc42ae41ec28bc1', 'tests/test_motion_action.py': 'e5a457199b9f2d1c886d3fd463d3ed99dff96a7340b1966db46ce8fa02a9dc90', 'tests/test_prepare_custom_videos.py': 'dd2b5a67eb188a5e29a66e6fb61055f6410f44d5910285fbfb9a8663f6356d7d', 'tests/test_prepare_reviewed_protocol.py': '6e81a5738d540641105bb9158da2f998fc4ce97a8b8bb0d4dd6bbacc355895f7', 'tests/test_prepare_visual_framepack.py': '630e95f7c17b3edba06154e4775b4421bdd47f6a4fd704533460217b25556114', 'tests/test_privileged_notes.py': '89456a4f0e920ffd3038f92556271ad44582eecacc536efd242d6d3ca37bc233', 'tests/test_reviewed_models.py': '04b834cdaa3b793a12d0786ed6bba26ead069a203150f336b48248d5159905c8', 'tests/test_reviewed_scene_groups.py': '50a842019ab9744dff12cefa8b20ed0529234ba739b626ddb7601587ffdb38e7', 'tests/test_scene_temporal.py': '22fdf1d32950c0dfef5a01c6844fd4134f0a0122b95de30573ab133d8acbaa61', 'tests/test_semantic_clip.py': '6877a5a7fb6a7c84a9f1479e3ac5bf355fd685e01cd4a03182841a307132b60f', 'tests/test_semantic_heads.py': 'bb9897ff8fd576220595597cfdd9c367fdaa6f7d7b341253b934c77802f58602', 'tests/test_semantic_predict.py': '2e5864742f029f535832dacb4fcec0529754feb423e1df8e23f2432258ef8fab', 'tests/test_semantic_vlm.py': '7cf60569a6f8db404da520432dcf1aab55bb0a8470f67e2201e3f3c7f44fc41c', 'tests/test_temporal.py': 'e5adca1d202585b480b2382d602ea076a909fec9ecc11f6c55363543a37db65f', 'tests/test_unlabeled_adaptation.py': '65da6d8e6ae982d76c85be6c37023b834fe3ec5e09fb4e2aa863a3b776fb1ed0', 'tests/test_verify_custom_labels.py': '5f466fb3a7be094cb31a68689859e937fdeba665f4de940f7f1e19b9181c90ad', 'tools/audit_preprocessing.py': '2b601326edae5d4b090a0641cce0f4f58e189d1640786a87b0baafb0267da66c', 'tools/benchmark_custom_annotation_detail.py': 'd0be69f25ab0ad9d9288c0cf51d770aa91fbeef43eb5268caf8c5af8ab7d53b1', 'tools/benchmark_semantic_vlm.py': '9532bcea81ecf15058f1adc34ccc360eb5b9511dc265e07d35bc81a41c6f8aa5', 'tools/build_boosted_notebook.py': 'ade1abd0a8c1118a647798ad9f1f5eb856e3518faa178a48ae5306aed42df155', 'tools/build_custom_manifest.py': 'ca2490a1c749015581e1b126d4419f7643c70427ae282cf9949ce50f008461a6', 'tools/build_custom_notebook.py': 'ef69b7902aa3944a484c2db0d879f34525b4181160dc01fd0d058a8a1a11850a', 'tools/build_notebook.py': '665bb7e3624d56989cc5c8ce5c2b749af8cf971d8fc42b73260e93bd9077cc54', 'tools/build_reviewed_notebook.py': '9c8647bb2235b80d2e92f96dbe94cc19b31a278c4cbd8deac2a82268793001c7', 'tools/build_scene_temporal_notebook.py': 'fc170eaf541bbc56e31a3424359c4e911f2d0ca4adf38198cd41ff92f042f0bf', 'tools/build_temporal_notebook.py': '827e89723d036057de93c6a2ffdc2bddda9f664edc798c630b2abb723046dee1', 'tools/build_visual_notebook.py': '5d2d81acc9060d85b652fb1a0b2aac79a440892befa6256ada2abae689a9bfa1', 'tools/diagnose_privileged_notes.py': '95a6fac2e83973c681500463e53648fe55fd2868a24b946ab1613cb0d3e8efee', 'tools/export_custom_results.py': 'ff16fc1c336e37831589b6fccf047e8fda0d369daeffde48c396389d3888c7a0', 'tools/extract_reviewed_features.py': 'e2d78cc795393af534819a003d1339f68ad119a6bb50f3370b7c34b101b8a146', 'tools/ingest_reviewed_workbook.py': '974651e0ead423bc2c81a19789a8ea50e8e6fd1f30995f921e9746ff1c23fe82', 'tools/make_reviewed_numeric_archive.py': 'd394fc1a00974784d345c32f99f2da00e9a675214c5bb6b5816f3da36ccea71b', 'tools/match_custom_cameras.py': '68372966ea76b464d643d54425c4c960740c746ffc846a99106eda212b769bf9', 'tools/prepare_custom_videos.py': '13580fe251a04bda491f0d4c958557c6bf9b3df23658c461648cb4dfd60e1b6e', 'tools/prepare_privileged_notes.py': '9913076dc8f983469cecc8b73f3931636c68e8d03005b92d2b8a0a9f72f029d9', 'tools/prepare_reviewed_protocol.py': '4086cddc6f4402c95c1b0419d7c42fec1ad54e6f51368a54b9944783fc673f89', 'tools/prepare_visual_framepack.py': 'e300f3cc98e6c80eff024c07d0799ef401ccf207cd5ffc1aa4eff3500089a8d4', 'tools/propose_event_review.py': '242dcf17719aeb0afe5d7a2430fe581cd77f1b4f93da7301b898b422f62d3b30', 'tools/review_annotations.py': '23bb5c6204ef907463b0efc0c15f7dd408b15aaec9124fc9243e1c7c08fe18e7', 'tools/reviewed_scene_groups.py': '8e99aed7963d18951028351abd3c33f53620262f02b3b523701a357384d3a80a', 'tools/run_custom_colab.py': '41c87a40b83e822c4f58df2fd2895edd097bcf22d60473ec2019d3f310695615', 'tools/run_scene_temporal_colab.py': '84d81174844df99108469af6ef2d996974a21135d928ceca600f2913118e2a71', 'tools/run_temporal_colab.py': 'd8902e12de792da458bf8b2cfb9ff243e22b15ad8360d0b9efef1d49074267b2', 'tools/summarize_boosted.py': 'adb6211493f4ca1afa13f29ccf225f3b40f7f27d1fc0bdf9a560074365d98867', 'tools/summarize_custom.py': 'a8980388bd05cb58847103c44d2b6851d4ffc6ff1b385978a6f68a3049f98804', 'tools/summarize_reviewed.py': 'ac7f967e2709046569dcaa795db48e0699c87f50e0250f5ee845b535ab91a952', 'tools/summarize_scene_temporal.py': 'd79f5024b1fee08bf24c90f236f43fe54d502b242561247384093b0b49b857b6', 'tools/summarize_temporal.py': 'e4705c652e5c0f7c7fab05afa7b4b0eb2e7b9c38ffc936912cbcfdc999e99eec', 'tools/verify_custom_labels.py': '3ba2cad791c444d571a776573a1be80270cc1645bb53e3059ca452c2478e3164'}
def digest_bytes(value): return hashlib.sha256(value).hexdigest()
def source_root_valid(directory):
    return all((directory/name).is_file() and digest_bytes((directory/name).read_bytes()) == digest
        for name,digest in SOURCE_MEMBERS.items())
roots = [path.parent for path in INPUT.rglob('finetune_experiment.py')
    if source_root_valid(path.parent)]
source_archive = next((path for path in INPUT.rglob('visual_finetune_source.zip')
    if digest_bytes(path.read_bytes()) == 'ecd6ed18cad9d6a201d0ce9951281249597b8521b4f362c9a936555bc49d321f'), None)
if roots:
    assert len(roots) == 1, 'Multiple source roots: select one exact private input'
    for name,digest in SOURCE_MEMBERS.items():
        target = WORKSPACE/name
        assert target.resolve().is_relative_to(WORKSPACE.resolve())
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes((roots[0]/name).read_bytes())
else:
    embedded = '...'
    assert source_archive is not None or embedded, 'Attach the private source input, or use the self-contained notebook'
    payload = source_archive.read_bytes() if source_archive is not None else __import__('base64').b64decode(embedded)
    with zipfile.ZipFile(io.BytesIO(payload)) as archive:
        for name,digest in SOURCE_MEMBERS.items():
            value = archive.read(name)
            assert digest_bytes(value) == digest
            target = WORKSPACE/name
            assert target.resolve().is_relative_to(WORKSPACE.resolve())
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(value)
builder64 = None
if builder64:
    target = WORKSPACE/'tools/build_visual_notebook.py'
    target.write_bytes(__import__('base64').b64decode(builder64))
assert source_root_valid(WORKSPACE)
assert digest_bytes((WORKSPACE/'datathon_ml/finetune_visual.py').read_bytes()) == 'a1bff5b688873ce11b8ee668fb9648d92f5276b082d29a5361f1d181777dd7ce'
os.chdir(WORKSPACE)
if str(WORKSPACE) not in sys.path: sys.path.insert(0, str(WORKSPACE))
RUN = WORKSPACE/'runs/visual_finetune_v2'
RUN.mkdir(parents=True, exist_ok=True)
(RUN/'source_restore_receipt.json').write_text(json.dumps({'verified_source_members':len(SOURCE_MEMBERS),
    'original_uploaded_source_zip_sha256':'ecd6ed18cad9d6a201d0ce9951281249597b8521b4f362c9a936555bc49d321f', 'finetune_module_sha256':'a1bff5b688873ce11b8ee668fb9648d92f5276b082d29a5361f1d181777dd7ce'},indent=2))
print({'verified_source_files':len(SOURCE_MEMBERS),'workspace_ready':True})


In [ ]:
# Public setup only. A previously downloaded official checkpoint is reused.
from importlib.metadata import version, PackageNotFoundError
try: ready=version('open_clip_torch')=='3.2.0'
except PackageNotFoundError: ready=False
if not ready:
    subprocess.run([sys.executable,'-m','pip','install','-q','open_clip_torch==3.2.0',
        '-r',str(WORKSPACE/'requirements-ml.txt')],check=True)
import torch
assert torch.cuda.is_available(), 'Enable a GPU runtime before this visual fine-tuning experiment'
from datathon_ml.clip_features import ClipFeatureConfig,_resolve_clip_weights,EXPECTED_WEIGHT_SHA256
available=[globals().get('BASE_WEIGHTS'),globals().get('CHECKPOINT'),BASE/'clip_openai_ViT-B-32.pt']
available=[Path(path) for path in available if path is not None and Path(path).is_file()
    and digest_bytes(Path(path).read_bytes())==EXPECTED_WEIGHT_SHA256]
CHECKPOINT=available[0] if available else _resolve_clip_weights(ClipFeatureConfig(allow_download=True),torch)
assert digest_bytes(CHECKPOINT.read_bytes())==EXPECTED_WEIGHT_SHA256
print({'gpu_count':torch.cuda.device_count(),'gpu_name':torch.cuda.get_device_name(0),
    'base_checkpoint_verified':True,'checkpoint_bytes':CHECKPOINT.stat().st_size})


Verify private input checksums and restore only train/validation images. Do not publish or preview the private framepack.

In [ ]:
# Verify every private member; materialize train/validation frames only.
FRAME_MANIFEST_SHA = '744a2b03b6cf4d8c4dab14bc2a019c190af3d606ce3874bd0aca1a0b3e1c4443'
FRAME_ZIP_SHA = '43a04bb8ae5e8cc34b29c86dc828e614019f2a513093aa5f2376c1e1cd2b4fcf'
FRAME_ARCHIVE = next((path for path in INPUT.rglob('reviewed_visual_framepack.zip')
    if digest_bytes(path.read_bytes()) == FRAME_ZIP_SHA), None)
extracted = [path.parent for path in INPUT.rglob('FRAMEPACK_MANIFEST.json')
    if digest_bytes(path.read_bytes()) == FRAME_MANIFEST_SHA]
if FRAME_ARCHIVE is not None:
    FRAME_ZIP_HANDLE=zipfile.ZipFile(FRAME_ARCHIVE)
    manifest_payload=FRAME_ZIP_HANDLE.read('FRAMEPACK_MANIFEST.json')
elif extracted:
    assert len(extracted)==1, 'Multiple matching private framepack roots'
    FRAME_INPUT_ROOT = extracted[0]
    manifest_payload = (FRAME_INPUT_ROOT/'FRAMEPACK_MANIFEST.json').read_bytes()
else:
    raise FileNotFoundError('Attach the separate private reviewed_visual_framepack.zip input')
assert digest_bytes(manifest_payload) == FRAME_MANIFEST_SHA
FRAME_INVENTORY = json.loads(manifest_payload)
FRAME_CHECKSUMS = FRAME_INVENTORY['members_sha256']
assert FRAME_INVENTORY['recordings']==58 and FRAME_INVENTORY['time_samples']==1879
assert FRAME_INVENTORY['jpeg_images']==3758 and FRAME_INVENTORY['contains_audio'] is False
if FRAME_ARCHIVE is not None:
    names=FRAME_ZIP_HANDLE.namelist()
    assert len(names)==len(set(names)) and set(names)==set(FRAME_CHECKSUMS)|{'FRAMEPACK_MANIFEST.json'}
def read_private_member(name):
    assert name in FRAME_CHECKSUMS
    parts=Path(name).parts
    assert not Path(name).is_absolute() and '..' not in parts
    assert name in {'dataset/manifest.json','dataset/split.json'} or (
        len(parts)==3 and parts[0]=='frames' and len(parts[1])==64 and
        (parts[2].endswith('.jpg') or parts[2] in {'metadata.json','_integrity.json'}))
    if FRAME_ARCHIVE is not None:
        value=FRAME_ZIP_HANDLE.read(name)
    else:
        path=(FRAME_INPUT_ROOT/name).resolve()
        assert path.is_relative_to(FRAME_INPUT_ROOT.resolve())
        value=path.read_bytes()
    assert digest_bytes(value)==FRAME_CHECKSUMS[name], 'Private framepack member checksum mismatch'
    return value
# Read bytes for checksum verification only; no images, captions or audio enter outputs.
for name in FRAME_CHECKSUMS: read_private_member(name)
records=json.loads(read_private_member('dataset/manifest.json'))
split=json.loads(read_private_member('dataset/split.json'))
assert len(records)==len({row['sha256'] for row in records})==58
assert [len(split[name]) for name in ('train','validation','test')]==[34,12,12]
from datathon_ml.reviewed_models import validate_split
validate_split(records,split)
assert split['protocol']['type']=='camera_exclusive_source_overlap_development'
for name in ('dataset/manifest.json','dataset/split.json'):
    target=WORKSPACE/name; target.parent.mkdir(parents=True,exist_ok=True); target.write_bytes(read_private_member(name))
assert all(not (WORKSPACE/'frames'/key).exists() for key in split['test']), 'Use a fresh fit workspace; former-test frames are already materialized'
fit_keys=set(split['train'])|set(split['validation'])
for name in FRAME_CHECKSUMS:
    parts=Path(name).parts
    if parts[0]=='frames' and parts[1] in fit_keys:
        target=WORKSPACE/name; target.parent.mkdir(parents=True,exist_ok=True); target.write_bytes(read_private_member(name))
(RUN/'framepack_restore_receipt.json').write_text(json.dumps({'framepack_zip_sha256':FRAME_ZIP_SHA,
    'framepack_manifest_sha256':FRAME_MANIFEST_SHA,'verified_member_count':len(FRAME_CHECKSUMS),
    'records':58,'train':34,'validation':12,'exposed_development':12,
    'former_test_frame_directories_in_fit_workspace':sum((WORKSPACE/'frames'/key).exists() for key in split['test']),
    'former_test_is_independent':False,'audio_used':False},indent=2))
print({'verified_private_members':len(FRAME_CHECKSUMS),'fit_framepacks':len(fit_keys),'former_test_materialized':0})


In [ ]:
# All subsequent subprocesses block network access, including inference.
from tools.run_scene_temporal_colab import run_guarded
COMMON=['--manifest','dataset/manifest.json','--split','dataset/split.json',
    '--frame-root','frames','--checkpoint',str(CHECKPOINT),
    '--originals-from','reference/clip_baseline_cv.json','--output-dir',str(RUN),
    '--device','cuda','--threads','2']
run_guarded(WORKSPACE,RUN,'focused_tests','unittest',
    ['discover','-s','tests','-p','test_finetune_visual.py','-v'],600)


Run two fixed supervised visual fits. Only training pixels/anchors update weights; validation alone selects the final candidate and threshold.

In [ ]:
# Two predeclared visual fits; no former-test frames in this fit workspace.
assert all(not (WORKSPACE/'frames'/key).exists() for key in split['test'])
run_guarded(WORKSPACE,RUN,'fit','finetune_experiment',['fit',*COMMON],3600)
selection=json.loads((RUN/'visual_selection.json').read_text())
selection_lock=json.loads((RUN/'visual_selection_lock.json').read_text())
BEST_MODEL=RUN/selection['winner']['artifact']
assert digest_bytes(BEST_MODEL.read_bytes())==selection_lock['winner_sha256']
assert digest_bytes((RUN/'visual_selection.json').read_bytes())==selection_lock['selection_sha256']
assert selection['development_partition_opened_for_selection'] is False
print('Frozen validation-selected visual candidate:',selection['winner']['candidate'])
print('Validation metrics:',json.dumps(selection['winner']['validation_metrics'],indent=2))


Score the already exposed development partition after the validation winner is saved and hashed.

In [ ]:
# Release the formerly exposed development partition only after selection.
selection_lock=json.loads((RUN/'visual_selection_lock.json').read_text())
assert digest_bytes(BEST_MODEL.read_bytes())==selection_lock['winner_sha256']
assert digest_bytes((RUN/'visual_selection.json').read_bytes())==selection_lock['selection_sha256']
for name in FRAME_CHECKSUMS:
    parts=Path(name).parts
    if parts[0]=='frames' and parts[1] in set(split['test']):
        target=WORKSPACE/name; target.parent.mkdir(parents=True,exist_ok=True); target.write_bytes(read_private_member(name))
(RUN/'development_frame_release.json').write_text(json.dumps({'released_sources':sorted(split['test']),
    'selection_lock_sha256':digest_bytes((RUN/'visual_selection_lock.json').read_bytes()),
    'winner_sha256':selection_lock['winner_sha256'],'independent_test':False},indent=2))
run_guarded(WORKSPACE,RUN,'score','finetune_experiment',['score',*COMMON,'--model',str(BEST_MODEL)],1800)
report=json.loads((RUN/'visual_development_report.json').read_text())
assert report['independent_test'] is False
print('Previously exposed mixed development metrics:',json.dumps(report['metrics'],indent=2))
print('Per-domain metrics:',json.dumps(report['domain_metrics'],indent=2))
print('Development accuracy >80%:',report['metrics']['accuracy']>.80)
print('Development accuracy >=87%:',report['metrics']['accuracy']>=.87)
print('These scores do not establish accuracy on unseen filming sessions.')


In [ ]:
# Native frame-pack prediction receives no target labels or point annotations.
source_sha=sorted(split['train'])[0]
run_guarded(WORKSPACE,RUN,'inference','finetune_experiment',['predict',
    '--frame-root','frames','--source-sha256',source_sha,'--checkpoint',str(CHECKPOINT),
    '--model',str(BEST_MODEL),'--output-dir',str(RUN/'inference'),'--device','cuda'],600)
print('Fresh-process frame-pack inference includes checkpoint loading, excludes original video decoding.')


In [ ]:
# Independently recompute classification/localization and verify saved ownership.
from datathon_ml.model import classification_metrics,_localization_metrics
rows=report['predictions']
assert {row['sha256'] for row in rows}==set(split['test']) and len(rows)==len(split['test'])
targets={row['sha256']:row for row in records}
for row in rows:
    expected=targets[row['sha256']]
    assert row['label']==expected['label'] and row['event_times']==expected['event_times']
    assert row['group_id']==expected['group_id']
recomputed=classification_metrics([r['label'] for r in rows],[r['score'] for r in rows],[r['prediction'] for r in rows])
for key,value in recomputed.items():
    if isinstance(value,float): assert abs(value-report['metrics'][key])<1e-10
    else: assert value==report['metrics'][key]
assert _localization_metrics(rows)==report['metrics']['localization']
assert report['model_sha256']==digest_bytes(BEST_MODEL.read_bytes())
for row in selection['leaderboard']:
    artifact=torch.load(RUN/row['artifact'],map_location='cpu',weights_only=True)
    assert set(artifact['training_sha256'])==set(split['train'])
    assert not set(artifact['frame_pack_metadata_sha256'])&set(split['test'])
    assert artifact['source_sha256']['finetune_visual']=='a1bff5b688873ce11b8ee668fb9648d92f5276b082d29a5361f1d181777dd7ce'
runtimes={name:json.loads((RUN/f'{name}_runtime.json').read_text()) for name in ('focused_tests','fit','score','inference')}
assert all(value['completed'] and value['network_attempts']==0 for value in runtimes.values())
summary={'evaluation':report['evaluation'],'independent_test':False,'metrics':report['metrics'],
    'domain_metrics':report['domain_metrics'],'validation_metrics':selection['winner']['validation_metrics'],
    'selected_candidate':selection['winner']['candidate'],'threshold':selection['winner']['threshold'],
    'candidate_count':len(selection['leaderboard']),'training_recordings':34,'validation_recordings':12,
    'previously_exposed_development_recordings':12,'source_sha256':selection['source_sha256'],
    'runtimes':runtimes,'labels_groups_predictions_metrics_and_artifact_verified':True,
    'new_session_accuracy_established':False,'hosted_llm_api_used':False,'audio_used':False}
(RUN/'visual_experiment_summary.json').write_text(json.dumps(summary,indent=2))
print('Verified visual development accuracy:',summary['metrics']['accuracy'])


Export exact source, fine-tuned parameter deltas, numeric reports and a clean notebook. Raw frames and the public base checkpoint are excluded.

In [ ]:
# Export model deltas, numeric reports, source and a clean notebook; no frames.
builder=WORKSPACE/'tools/build_visual_notebook.py'
if not builder.is_file(): builder.write_bytes(__import__('base64').b64decode('...'))
source_input=RUN/'source_snapshot_input.zip'
with zipfile.ZipFile(source_input,'w',zipfile.ZIP_DEFLATED,compresslevel=9) as archive:
    for name,digest in SOURCE_MEMBERS.items():
        value=(WORKSPACE/name).read_bytes(); assert digest_bytes(value)==digest
        archive.writestr(name,value)
subprocess.run([sys.executable,str(builder),'--source-zip',str(source_input),
    '--frame-manifest-sha',FRAME_MANIFEST_SHA,'--notebook',str(WORKSPACE/'datathon_visual.ipynb'),
    '--bundle-out',str(RUN/'export_source.zip')],cwd=WORKSPACE,check=True)
notebook=json.loads((WORKSPACE/'datathon_visual.ipynb').read_text())
assert all(cell.get('outputs',[])==[] and cell.get('execution_count') is None for cell in notebook['cells'] if cell['cell_type']=='code')
members={'datathon_visual.ipynb':(WORKSPACE/'datathon_visual.ipynb').read_bytes()}
with zipfile.ZipFile(RUN/'export_source.zip') as archive:
    for name in archive.namelist():
        if name.endswith('.py'): assert archive.read(name)==(WORKSPACE/name).read_bytes()
        members['source/'+name]=archive.read(name)
for path in RUN.rglob('*'):
    if path.is_file() and path.suffix in {'.pt','.json','.log'}:
        relative=path.relative_to(WORKSPACE)
        assert not set(relative.parts)&{'dataset','frames','features','open'}
        members[relative.as_posix()]=path.read_bytes()
assert not any(name.endswith(('.jpg','.jpeg','.png','.mp4','.mov','.xlsx')) for name in members)
members['EXPORT_MANIFEST.json']=json.dumps({'contains_private_frames':False,'contains_raw_media':False,
    'contains_base_clip_checkpoint':False,'contains_supervised_visual_parameter_deltas':True,
    'members_sha256':{name:digest_bytes(value) for name,value in sorted(members.items())}},indent=2).encode()
RESULTS_ZIP=BASE/'datathon_visual_results.zip'
with zipfile.ZipFile(RESULTS_ZIP,'w',zipfile.ZIP_DEFLATED,compresslevel=5) as archive:
    for name,value in sorted(members.items()): archive.writestr(name,value)
print({'export_bytes':RESULTS_ZIP.stat().st_size,'export_sha256':digest_bytes(RESULTS_ZIP.read_bytes()),
    'export_members':len(members),'contains_private_frames':False})
from IPython.display import FileLink,display
display(FileLink(str(RESULTS_ZIP)))
if not Path('/kaggle/working').is_dir() and Path('/content').is_dir():
    from google.colab import files
    files.download(str(RESULTS_ZIP))
